# Gift-Eval Head Ablation Analysis

This notebook analyzes the impact of **attention head ablation** on Time Series Foundation Model (TSFM) performance across the Gift-Eval benchmark datasets.

## Overview

**Purpose:** Evaluate model robustness by progressively ablating (zeroing out) attention heads and measuring the effect on forecasting metrics.

**Supported Models:** TimesFM 2.5, Chronos Bolt, Toto

## Analysis Pipeline

1. **Data Loading** — Load ablation results across multiple random seeds and select the best-performing seed per ablation level
2. **Normalization** — Optionally normalize metrics by seasonal naive baseline for fair comparison
3. **Statistical Analysis** — Compute geometric means and standard errors across datasets for each ablation level
4. **Visualization:**
   - **Line plots** — Performance degradation curves as heads are ablated per layer
   - **Bar plots** — Dataset-level comparison of baseline vs. best ablation performance  
   - **Heatmaps** — Per-dataset performance across all ablation levels
5. **Dataset Categorization** — Identify resilient, improved, and vulnerable datasets based on ablation response

## Key Metrics
- Primary: MASE (Mean Absolute Scaled Error) at 0.5 quantile
- Aggregation: Geometric mean across datasets

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tsfm_lens.utils import apply_custom_style, normalize_by_seasonal_naive
from scipy.stats import gmean
import pprint
from collections import defaultdict

#### Configuration

In [ ]:
model_name = "Toto"

#### Presets

In [ ]:
model_to_dir_mapping = {
    "TimesFM 2.5": "google-timesfm-2.5-200m-pytorch",
    "Chronos Bolt": "amazon-chronos-bolt-base",
    "Chronos": "amazon-chronos-t5-base",
    "Toto": "Datadog-Toto-Open-Base-1.0_samples-20",
}

# Select the metric to analyze
SELECTED_METRIC = "eval_metrics/MASE[0.5]"
# SELECTED_METRIC = "eval_metrics/sMAPE[0.5]"

metric_pretty_name = "".join(c for c in SELECTED_METRIC.split("/")[-1] if c.isalpha())

print(f"Analyzing metric: {SELECTED_METRIC}")
print(f"Metric name: {metric_pretty_name}")


In [ ]:
num_layers_mapping = {
    "TimesFM 2.5": 20,
    "Chronos Bolt": 12,
    "Toto": 12,
}

num_heads_mapping = {
    "TimesFM 2.5": 16,
    "Chronos Bolt": 12,
    "Toto": 12,
}

num_layers = num_layers_mapping[model_name]
num_heads = num_heads_mapping[model_name]

In [ ]:
apply_custom_style("../../config/plotting.yaml")
HOME_DIR = os.getenv("HOME")
root_dir = os.path.join(HOME_DIR, "tsfm-lens")  # type: ignore

save_figs = True
figs_save_dir = os.path.join(root_dir, "figs", f"gift-eval_{metric_pretty_name}", model_to_dir_mapping[model_name])
if not os.path.exists(figs_save_dir):
    os.makedirs(figs_save_dir)
if save_figs:
    print(f"Saving figures to: {figs_save_dir}")

## Load Data


In [ ]:
MODEL_CONFIGS = {
    "TimesFM 2.5": {
        "layers": list(range(20)),
        "head_levels": list(range(1, 16)),
        # "head_levels": [n for n in range(1, 16) if n not in [14, 15]],
        # "head_levels": [1, 2, 4, 6, 8, 10, 12, 14],
        "n_heads": 16,
    },
    "Chronos Bolt": {
        "layers": range(12),
        "head_levels": range(1, 12),
        "n_heads": 12,
    },
    "Toto": {
        "layers": range(12),
        "head_levels": range(1, 12),
        "n_heads": 12,
    },
}

if model_name not in MODEL_CONFIGS:
    raise ValueError(f"Model {model_name} not supported yet")

cfg = MODEL_CONFIGS[model_name]
selected_layers_lst, levels_num_heads_ablated, n_divisor = cfg["layers"], cfg["head_levels"], cfg["n_heads"]

# Build ablation files dictionary for all selected layers
ablation_meaning_str = "Ablation of Heads Across Layers"
ablation_level_meaning_str = "Pct of Heads Ablated per Layer"
ablation_level_meaning_alternative_str = "Number of Heads Ablated per Layer"

ablated_files_dict, ablation_meaning_str_dict = {}, {}
ablation_level_meaning_str_dict, ablation_level_meaning_alternative_str_dict = {}, {}

for layer in selected_layers_lst:
    key = f"Layer {layer}" if not isinstance(layer, list) else f"Layers {'-'.join(map(str, layer))}"
    ablation_meaning_str_dict[key] = f"Ablation of Heads in Layer {layer}"
    ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated"
    ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated"
    ablated_files_dict[key] = {
        0: "original_all_results.csv",
        **{n: f"head_layers_{layer}_heads-{n}_all_results.csv" for n in levels_num_heads_ablated},
        n_divisor: f"head_layers_{layer}_heads-all_all_results.csv",
    }

In [ ]:
pprint.pp(ablated_files_dict)

In [ ]:
srank_rseeds_lst = srank_reverse_rseeds_lst = [42, 99]

# Build metrics directories for each selection strategy
base_dir = os.path.join(root_dir, "results", model_to_dir_mapping[model_name])
make_dirs = lambda seeds, prefix="": {s: os.path.join(base_dir, f"{prefix}rseed-{s}") for s in seeds}

srank_metrics_dir_dict = make_dirs(srank_rseeds_lst, "srank_")
srank_reverse_metrics_dir_dict = make_dirs(srank_reverse_rseeds_lst, "srank_reverse_")

metrics_dir_dict_mapping = {
    "Stable Rank (low to high)": srank_metrics_dir_dict,
    "Stable Rank (high to low)": srank_reverse_metrics_dir_dict,
}

for name, d in metrics_dir_dict_mapping.items():
    print(f"{name} head selection metrics directory: {d}")


In [ ]:
ablated_files_dict.keys()

In [ ]:
# Load CSVs, compute gmean per rseed, select best rseed per level
gmean_key = f"gmean_{SELECTED_METRIC}"
best_filepaths_by_key = {}
data_by_key = defaultdict(dict)


for head_selection_strategy, metrics_dir_dict in metrics_dir_dict_mapping.items():
    for layer_idx, ablation_files in ablated_files_dict.items():
        print(f"Processing layer key: {layer_idx}")
        best_filepaths_by_key[layer_idx], curr_data = {}, {}

        for level, filepath in ablation_files.items():
            # Load valid data for each rseed
            rseed_data = {}
            for rseed, metrics_dir in metrics_dir_dict.items():
                df_path = os.path.join(metrics_dir, filepath)
                if not os.path.exists(df_path):
                    # print(f"Level {level}, rseed {rseed}: Skipping (not found)")
                    continue
                df = pd.read_csv(df_path)
                if len(df) != 97:
                    # print(f"Level {level}, rseed {rseed}: Skipping (incomplete)")
                    continue
                print(f"Level {level}, rseed {rseed}: Loading")
                df["ablation_level"] = level
                rseed_data[rseed] = (df, df_path)

            if not rseed_data:
                # print(f"Level {level}: No data found, skipping")
                continue

            # Select rseed with lowest geometric mean
            rseed_gmeans = {r: gmean(df[SELECTED_METRIC]) for r, (df, _) in rseed_data.items()}
            best_rseed = min(rseed_gmeans, key=lambda r: rseed_gmeans[r])
            best_df, best_path = rseed_data[best_rseed]

            best_filepaths_by_key[layer_idx][level] = {
                "best_rseed": best_rseed,
                "best_filepath": best_path,
                "available_rseeds": list(rseed_data.keys()),
                gmean_key: rseed_gmeans[best_rseed],
                "num_rseeds_evaluated": len(rseed_data),
            }
            curr_data[level] = best_df
            print(
                f"Level {level}: Best rseed={best_rseed} (gmean={rseed_gmeans[best_rseed]:.6f}) from {len(rseed_data)} rseeds"
            )

        if curr_data:
            data_by_key[head_selection_strategy][layer_idx] = pd.concat(curr_data.values(), ignore_index=True)
            print(
                f"\nTotal: {len(data_by_key[head_selection_strategy][layer_idx])} rows, "
                f"{data_by_key[head_selection_strategy][layer_idx]['dataset'].nunique()} unique datasets"
            )
        else:
            print(f"\nNo valid data found for {head_selection_strategy} - {layer_idx}")

## Normalize by Seasonal Naive Baseline


In [ ]:
# Configuration: Set to True to normalize metrics by seasonal naive baseline
NORMALIZE_BY_SEASONAL_NAIVE = True
is_normalized = False

print(f"Normalization by seasonal naive baseline: {NORMALIZE_BY_SEASONAL_NAIVE}")


In [ ]:
data_by_key.keys()

In [ ]:
data_by_key["Stable Rank (low to high)"].keys()

In [ ]:
if NORMALIZE_BY_SEASONAL_NAIVE and not is_normalized:
    seasonal_naive_df = pd.read_csv(os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv"))
    print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets. Normalizing...")
    for head_selection_strategy in data_by_key:
        for layer_idx in data_by_key[head_selection_strategy]:
            data_by_key[head_selection_strategy][layer_idx] = normalize_by_seasonal_naive(
                data_by_key[head_selection_strategy][layer_idx], seasonal_naive_df
            )
    is_normalized = True
else:
    print("Skipping normalization.")


### Display Available Metrics


In [ ]:
# Display available metrics
metric_columns = [col for col in df.columns if "eval_metrics" in col]
print("Available metrics:")
for i, metric in enumerate(metric_columns, 1):
    print(f"{i}. {metric}")


### Line Plot


In [ ]:
data_by_key.keys()

In [ ]:
# Prepare box data and compute stats for nested structure: data_by_key[strategy][layer]
box_data_by_key = defaultdict(dict)
stats_by_level_by_key = defaultdict(dict)

agg_funcs = [
    ("median", "median"),
    ("mean", "mean"),
    ("geom_mean", lambda x: gmean(x)),
    ("geom_ste", lambda x: np.exp(np.std(np.log(x)) / np.sqrt(len(x)))),
    ("q25", lambda x: x.quantile(0.25)),
    ("q75", lambda x: x.quantile(0.75)),
]

for strategy, layer_data in data_by_key.items():
    for layer_key, df in layer_data.items():
        # Clean data
        box_data = df[["dataset", "ablation_level", SELECTED_METRIC]].replace([np.inf, -np.inf], np.nan).dropna()
        box_data_by_key[strategy][layer_key] = box_data
        # Compute stats
        stats_by_level_by_key[strategy][layer_key] = (
            box_data.groupby("ablation_level")[SELECTED_METRIC]
            .agg([f for _, f in agg_funcs])
            .set_axis([n for n, _ in agg_funcs], axis=1)
            .reset_index()
        )

In [ ]:
# single_layer_colors = plt.cm.get_cmap("tab20c").colors  # type: ignore
# single_layer_colors = [c for i, c in enumerate(plt.cm.get_cmap("tab20c").colors) if i % 4 != 3]  # type: ignore

if model_name == "TimesFM 2.5":
    single_layer_colors = plt.cm.get_cmap("tab20c").colors  # type: ignore
else:
    single_layer_colors = [c for i, c in enumerate(plt.cm.get_cmap("tab20c").colors) if i % 4 != 3]  # type: ignore

colors = list(single_layer_colors)
colors_dict = {k: v for k, v in zip(ablated_files_dict.keys(), colors)}
print(colors_dict.keys())

In [ ]:
# For each layer, select the head selection strategy with minimum geom_mean (best performance)
# Preserve original layer ordering from first strategy
all_layers = list(next(iter(stats_by_level_by_key.values())).keys())
best_stats_by_layer = {}
best_strategy_by_layer = {}

# layers_to_show = [0, 2, 3, 5, 6, 7, 8, 10, 11, 12, 13, 19]
num_tot_layers = num_layers_mapping[model_name]
layers_to_show = list(range(num_tot_layers))

for i, layer in enumerate(all_layers):
    if i not in layers_to_show:
        continue

    best_within_1pct, best_geom_mean, best_strategy = float("inf"), float("inf"), None

    for strategy, layer_stats in stats_by_level_by_key.items():
        if layer not in layer_stats or len(layer_stats[layer]) <= 1:
            continue

        # Find baseline (geom_mean at ablation_level 0)
        baseline_rows = layer_stats[layer].loc[layer_stats[layer]["ablation_level"] == 0, "geom_mean"]
        if len(baseline_rows) == 0:
            continue

        layer_baseline = baseline_rows.values[0]
        # Count how many ablation levels stay within 1% of baseline
        within_1pct = (layer_stats[layer]["geom_mean"] <= 1.01 * layer_baseline).sum()
        # Calculate mean geom_mean for tie-breaking
        mean_geom_mean = layer_stats[layer]["geom_mean"].mean()

        # Prefer strategy with more levels within 1% (higher is better, so negate for comparison)
        # If tie, use the strategy with the lower geometric mean
        if -within_1pct < best_within_1pct or (-within_1pct == best_within_1pct and mean_geom_mean < best_geom_mean):
            best_within_1pct, best_geom_mean, best_strategy = -within_1pct, mean_geom_mean, strategy

    if best_strategy:
        best_stats_by_layer[layer] = stats_by_level_by_key[best_strategy][layer]
        best_strategy_by_layer[layer] = best_strategy

print("Best strategy per layer:", best_strategy_by_layer)

# Create line plot with error bands
fig, ax = plt.subplots(figsize=(4, 5))


def add_ref_line(y, label, color, va, y_mult):
    ax.axhline(y=y, color=color, linestyle="--", linewidth=2, alpha=0.5, zorder=100)
    ax.text(0, y * y_mult, label, color=color, alpha=0.5, fontsize=8, fontweight="bold", va=va, ha="left", zorder=100)


baseline = None
for i, (layer, stats) in enumerate(best_stats_by_layer.items()):
    # Filter out failed ablation levels (those with geom_mean == 1.638366)
    stats = stats[~np.isclose(stats["geom_mean"], 1.638366)]

    if len(stats) == 0:
        continue

    x_pct = stats["ablation_level"] / n_divisor * 100
    strategy_short = best_strategy_by_layer[layer].replace("Stable Rank ", "").replace("(", "").replace(")", "")

    if baseline is None:  # Find baseline from first layer that has level 0
        baseline_rows = stats.loc[stats["ablation_level"] == 0, "geom_mean"]
        if len(baseline_rows) > 0:
            baseline = baseline_rows.values[0]
            add_ref_line(baseline, f"Baseline ({baseline:.3f})", "black", "top", 0.994)
            add_ref_line(1.01 * baseline, "Baseline + 1%", "black", "bottom", 1.004)

    ax.plot(
        x_pct,
        stats["geom_mean"],
        marker="^",
        markeredgecolor=colors_dict[layer],
        markeredgewidth=1,
        linewidth=3,
        markersize=0,
        color=colors_dict[layer],
        alpha=0.9,
        label=f"{layer} ({strategy_short})",
        zorder=i,
    )
    ax.fill_between(
        x_pct,
        stats["geom_mean"] * stats["geom_ste"] ** 0.2,
        stats["geom_mean"] / stats["geom_ste"] ** 0.2,
        alpha=0.08,
        color=colors_dict[layer],
        zorder=-1,
    )

if baseline is None:
    print("No valid data to plot")
else:
    ax.set(
        xlabel=f"{ablation_level_meaning_str} (%)",
        ylabel=f"{metric_pretty_name} (Geom Mean)",
        ylim=(0.98 * baseline, 1.1 * baseline),
    )
    ax.set_title(f"ZA Heads by Stable Rank ({model_name})", fontsize=10)
    for lbl in [ax.xaxis.label, ax.yaxis.label, ax.title]:
        lbl.set_fontweight("bold")
    ax.grid(True, alpha=0.2)
    legend = ax.legend(
        loc="upper left",
        frameon=True,
        fontsize=6,
        ncols=2,
        framealpha=1.0,
        handlelength=1.5,
        handletextpad=0.5,
        markerscale=0,
    )
    legend.set_zorder(1000)

    all_levels = {lvl for s in best_stats_by_layer.values() for lvl in s["ablation_level"]}
    xticks = sorted(x / n_divisor * 100 for x in all_levels)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{int(x)}" if x == int(x) else f"{x:.1f}" for x in xticks], fontsize=5.5)

    plt.tight_layout()
    if save_figs:
        save_path = os.path.join(figs_save_dir, f"{ablation_meaning_str}_line_plot_{model_name}_best_strategy.pdf")
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()

In [ ]:
failed_or_missing_by_layer = {}
for i, (layer, stats) in enumerate(best_stats_by_layer.items()):
    # Find failed ablation levels (those with geom_mean == 1.638366)
    matching_rows = stats[np.isclose(stats["geom_mean"], 1.638366)]
    failed_levels = set(matching_rows["ablation_level"].tolist())

    # Check for missing ablation levels
    all_expected_levels = set(range(0, n_divisor + 1))
    actual_levels = set(stats["ablation_level"].tolist())
    missing_levels = all_expected_levels - actual_levels

    # Union of failed and missing
    failed_or_missing = failed_levels | missing_levels
    if failed_or_missing:
        failed_or_missing_by_layer[layer] = failed_or_missing

print(failed_or_missing_by_layer)
tot_num_failed_or_missing = sum(len(levels) for levels in failed_or_missing_by_layer.values())
print(f"Total number of failed or missing ablation levels: {tot_num_failed_or_missing}")

In [ ]:
for i, (layer, stats) in enumerate(best_stats_by_layer.items()):
    print(f"========= Layer: {layer}")
    print(stats["ablation_level"])
    x_pct = stats["ablation_level"] / n_divisor * 100
    print(f"x_pct: {x_pct}")
    print(stats["geom_mean"])

In [ ]:
# Define heads@1pp as the (n_divisor - highest ablation level at or under baseline + 1%)
# This is the minimum number of heads needed for a specified layer that would have a relative performance within 1% of the baseline
# plot scatter plot of heads@1pp vs layer

# Get the baseline (same across all layers - the unablated model)
baseline = None
for layer, stats in best_stats_by_layer.items():
    baseline_rows = stats.loc[stats["ablation_level"] == 0, "geom_mean"]
    if len(baseline_rows) > 0:
        baseline = baseline_rows.values[0]
        break

if baseline is None:
    raise ValueError("Could not find baseline (ablation_level=0) in any layer")

threshold = baseline * 1.01  # 1% above baseline
print(f"Baseline: {baseline:.6f}")
print(f"Threshold (baseline + 1%): {threshold:.6f}")

# Calculate heads@1pp for each layer
heads_at_1pp = {}
for layer, stats in best_stats_by_layer.items():
    # Find the highest ablation level where geom_mean <= threshold
    valid_levels = stats[stats["geom_mean"] <= threshold]["ablation_level"]
    if len(valid_levels) > 0:
        highest_valid_ablation = valid_levels.max()
    else:
        # No ablation level is within 1% of baseline, need all heads
        highest_valid_ablation = 0

    # heads@1pp = total heads - highest ablation level that still works
    heads_at_1pp[layer] = n_divisor - highest_valid_ablation

print("\nheads@1pp per layer:")
for layer, heads in heads_at_1pp.items():
    stats = best_stats_by_layer[layer]
    valid_levels = stats[stats["geom_mean"] <= threshold]["ablation_level"]
    max_ablatable = valid_levels.max() if len(valid_levels) > 0 else 0
    print(f"  {layer}: {heads} heads (can ablate up to {max_ablatable} heads)")

# Extract layer numbers for plotting
layer_numbers = [int(layer.replace("Layer ", "")) for layer in heads_at_1pp.keys()]
heads_values = list(heads_at_1pp.values())

# Create scatter plot
fig, ax = plt.subplots(figsize=(5.5, 7))

# Color points by heads@1pp value (fewer heads = more redundant = better)
scatter = ax.scatter(
    layer_numbers,
    heads_values,
    c=heads_values,
    cmap="RdYlGn_r",  # Red for high (needs more heads), Green for low (more redundant)
    # cmap="RdBu",
    s=100,
    edgecolors="black",
    linewidths=1.5,
    vmin=0,
    vmax=n_divisor,
    zorder=10,
)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, shrink=0.8)
cbar.set_label("heads@1pp", fontsize=10, fontweight="bold")

# Reference lines
ax.axhline(y=n_divisor, color="red", linestyle="--", alpha=0.5, label=f"All heads ({n_divisor})")
ax.axhline(y=n_divisor / 2, color="orange", linestyle="--", alpha=0.5, label=f"Half heads ({n_divisor // 2})")

# Formatting
ax.set_xlabel("Layer", fontsize=12, fontweight="bold")
# ax.set_ylabel("heads@1pp (min heads for ≤1% degradation)", fontsize=11, fontweight="bold")
ax.set_title(f"{model_name}", fontsize=13, fontweight="bold")
ax.set_xticks(layer_numbers)
ax.tick_params(axis="x", labelsize=7)
ax.set_ylim(-0.5, n_divisor + 1)
ax.set_xlim(-0.5, max(layer_numbers) + 0.5)
# ax.legend(loc="center left", fontsize=9, frameon=True, framealpha=1.0)

# Add annotations for each point
for layer_num, heads in zip(layer_numbers, heads_values):
    ax.annotate(
        f"{heads}",
        (layer_num, heads),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9,
        fontweight="bold",
    )

plt.tight_layout()
if save_figs:
    save_path = os.path.join(figs_save_dir, f"heads_at_1pp_scatter_{model_name}.pdf")
    plt.savefig(save_path, bbox_inches="tight")
    print(f"\nSaved: {save_path}")

# Summary statistics
print(f"\n{'=' * 60}")
print(f"Summary Statistics for heads@1pp:")
print(f"  Mean: {np.mean(heads_values):.2f} heads")
print(f"  Median: {np.median(heads_values):.2f} heads")
print(
    f"  Min: {min(heads_values)} heads (most redundant: {list(heads_at_1pp.keys())[heads_values.index(min(heads_values))]})"
)
print(
    f"  Max: {max(heads_values)} heads (least redundant: {list(heads_at_1pp.keys())[heads_values.index(max(heads_values))]})"
)
print(f"  Total heads per layer: {n_divisor}")
print(f"  Average redundancy: {(1 - np.mean(heads_values) / n_divisor) * 100:.1f}% of heads can be ablated on average")


In [ ]:
# Calculate and display percentage change from baseline for each key
# Note: baseline (level 0) should be the same across all layers since it's the unablated model
for head_selection_strategy, selected_stats in stats_by_level_by_key.items():
    print(f"\nHead selection strategy: {head_selection_strategy}")
    cached_base_med, cached_base_gm = None, None  # Cache baseline - same for all layers

    for key, stats in selected_stats.items():
        baseline_rows = stats[stats["ablation_level"] == 0]
        using_cached = False

        if len(baseline_rows) > 0:
            base_med, base_gm = baseline_rows["median"].values[0], baseline_rows["geom_mean"].values[0]
            cached_base_med, cached_base_gm = base_med, base_gm  # Update cache
        elif cached_base_med is not None:
            base_med, base_gm = cached_base_med, cached_base_gm  # Use cached
            using_cached = True
        else:
            print(f"\n{key}: No baseline found yet, skipping")
            continue

        cached_note = " (using cached baseline)" if using_cached else ""
        print(f"\n% Change from Baseline for {key}{cached_note}:\n" + "=" * 70)
        for _, r in stats.iterrows():
            n, gm = int(r["ablation_level"]), r["geom_mean"]
            print(
                f"{n} heads ({n / n_divisor * 100:.1f}%): Median {(r['median'] / base_med - 1) * 100:+.2f}%, "
                f"Geom Mean {(gm / base_gm - 1) * 100:+.2f}% (curr: {gm:.3f})"
            )


### Get Combined Ablation Blueprint

In [ ]:
for i, (layer, stats) in enumerate(best_stats_by_layer.items()):
    h1pp = heads_at_1pp[layer]
    strategy_short = best_strategy_by_layer[layer].replace("Stable Rank ", "").replace("(", "").replace(")", "")
    # print(f"{layer} ({strategy_short}): {x_pct:.1f}%")
    print(f"{layer}: keep {h1pp} heads (ZA by srank {strategy_short})")

In [ ]:
ASSETS_DIR = os.path.join("../../", "assets")

In [ ]:
model_name

In [ ]:
model_to_circuitlens_mapping = {
    "Chronos": "CircuitLensChronos",
    "Chronos Bolt": "CircuitLensBolt",
    "Toto": "CircuitLensToto",
    "TimesFM 2.5": "CircuitLensTimesFM",
}

In [ ]:
strategy_short

In [ ]:
import json

model_srank_per_layer_path = os.path.join(
    ASSETS_DIR, f"{model_to_circuitlens_mapping[model_name]}_srank_low_to_high_by_layer.json"
)
# if not exists, raise error
if not os.path.exists(model_srank_per_layer_path):
    raise ValueError(f"Srank per layer file not found at {model_srank_per_layer_path}")
srank_per_layer = json.load(open(model_srank_per_layer_path))
heads_to_ablate = []

print(f"Total heads per layer: {n_divisor}")

for i, (layer, stats) in enumerate(best_stats_by_layer.items()):
    h1pp = heads_at_1pp[layer]
    num_heads_to_ablate = n_divisor - h1pp
    strategy_short = best_strategy_by_layer[layer].replace("Stable Rank ", "").replace("(", "").replace(")", "")
    print(f"{layer}: keep {h1pp} heads (ZA by srank {strategy_short})")

    layer_idx = int(layer.split(" ")[1])
    if strategy_short == "high to low":
        heads_to_ablate.extend(
            [
                (layer_idx, int(head))
                for head in list(srank_per_layer[str(layer_idx)].keys())[::-1][:num_heads_to_ablate]
            ]
        )
    elif strategy_short == "low to high":
        heads_to_ablate.extend(
            [(layer_idx, int(head)) for head in list(srank_per_layer[str(layer_idx)].keys())[:num_heads_to_ablate]]
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy_short}")

print(f"Ablating heads: {heads_to_ablate}")

In [ ]:
# # save heads_to_ablate to json file in ASSETS_DIR
# with open(
#     os.path.join(ASSETS_DIR, f"{model_to_circuitlens_mapping[model_name]}_ablate_for_heads_at_1pp.json"), "w"
# ) as f:
#     json.dump(heads_to_ablate, f)

## Identify Datasets close to Baseline

In [ ]:
selected_layer = "Layer 11"
head_selection_strategy = "Stable Rank (low to high)"
selected_layer_data = box_data_by_key[head_selection_strategy][selected_layer]
print(f"Selected layer: {selected_layer}")

# Get baseline and compute relative performance
baseline = selected_layer_data[selected_layer_data["ablation_level"] == 0][["dataset", SELECTED_METRIC]]
baseline.columns = ["dataset", "baseline_value"]
merged_data = selected_layer_data.merge(baseline, on="dataset")
merged_data["rel"] = merged_data[SELECTED_METRIC] / merged_data["baseline_value"]

In [ ]:
merged_data

In [ ]:
import seaborn as sns
from matplotlib.colors import Normalize

# Pivot and sort datasets by name/freq/term
# heatmap_data = box_data.pivot(index="dataset", columns="ablation_level", values=SELECTED_METRIC)
heatmap_data = merged_data.pivot(index="dataset", columns="ablation_level", values=SELECTED_METRIC)


def sort_key(name):
    parts = name.split("/")
    if len(parts) != 3:
        return (name, 0, 0)
    ds, freq, term = parts
    freq_ord = {"H": 1, "D": 2, "W": 3, "M": 4, "Q": 5, "Y": 6}
    freq_val = int(freq[:-1]) if freq.endswith("T") and freq[:-1].isdigit() else freq_ord.get(freq, 10) * 1000
    return (ds, freq_val, {"short": 0, "medium": 1, "long": 2}.get(term, 3))


heatmap_data = heatmap_data.reindex(sorted(heatmap_data.index, key=sort_key))
if len(heatmap_data) > 100:
    print(f"Showing top 50/{len(heatmap_data)} datasets")
    heatmap_data = heatmap_data.head(50)


# Compute max improving ablation level for each dataset
# Set tolerance: 0.0 = strictly <= baseline, 0.01 = within 1% increase of baseline, etc.
bold_tolerance = 0.01  # Change to 0.0 for strict <= baseline


def get_max_improving_level(row, tolerance=0.0):
    """For a row (dataset), find max ablation level where value <= baseline * (1 + tolerance)"""
    baseline = row.get(0, np.nan)
    if pd.isna(baseline):
        return None
    threshold = baseline * (1 + tolerance)
    improving = [lvl for lvl in row.index if lvl > 0 and row[lvl] <= threshold]
    return max(improving) if improving else 0


bold_levels = {ds: get_max_improving_level(heatmap_data.loc[ds], bold_tolerance) for ds in heatmap_data.index}

# Split into 3 parts and plot
n, vmin, vmax = len(heatmap_data), heatmap_data.min().min(), 3
splits = [heatmap_data.iloc[i * n // 3 : (i + 1) * n // 3] for i in range(3)]
# fig, axes = plt.subplots(1, 3, figsize=(20, max(8, n // 3 * 0.3)))
if model_name == "TimesFM 2.5":
    fig, axes = plt.subplots(1, 3, figsize=(26, max(8, n // 3 * 0.3)))

else:
    fig, axes = plt.subplots(1, 3, figsize=(22, max(8, n // 3 * 0.3)))

for i, (ax, data) in enumerate(zip(axes, splits)):
    sns.heatmap(data, annot=True, cmap="YlOrRd", vmin=vmin, vmax=vmax, cbar=False, linewidths=0.5, ax=ax)

    # Bold annotations for max improving ablation level per dataset
    for row_idx, ds in enumerate(data.index):
        best_lvl = bold_levels.get(ds)
        if best_lvl is not None and best_lvl in data.columns:
            col_idx = list(data.columns).index(best_lvl)
            text_idx = row_idx * len(data.columns) + col_idx
            if text_idx < len(ax.texts):
                ax.texts[text_idx].set_fontweight("bold")

    ax.set(xlabel=ablation_level_meaning_alternative_str, ylabel="Dataset" if i == 0 else "")
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontweight("bold")
    ax.set_title(f"Datasets {i * n // 3 + 1}-{i * n // 3 + len(data)}", fontsize=12, fontweight="bold")

# Shared colorbar and title
fig.subplots_adjust(right=0.92)

sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=fig.add_axes((0.94, 0.15, 0.015, 0.7)))
cbar.set_label(f"$\\mathbf{{{metric_pretty_name}}}$", fontsize=12, fontweight="bold")
# fig.suptitle(f"{ablation_meaning_str} ({model_name})", fontsize=20, fontweight="bold", y=0.96)
selected_layer_meaning_str = f"Ablations of Heads in {selected_layer}"
fig.suptitle(f"{selected_layer_meaning_str} ({model_name})", fontsize=20, fontweight="bold", y=0.96)

plt.tight_layout(rect=(0, 0, 0.93, 0.96))
if save_figs:
    save_path = os.path.join(figs_save_dir, f"{selected_layer_meaning_str}_datasets_heatmap_{model_name}.pdf")
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved: {save_path}")
plt.show()
print(f"Showing {n} datasets split into 3 columns")
